# TODO：研究タイトル

**G1：国家・世界経済の成長と資源配分の最適化**（補遺トピック）

チーム名：TODO　／　メンバー：TODO

> **補遺トピック G1・教育用の架空モデル**です。国名・経済指標・投資効果の係数はすべて架空で、現実の国家の GDP・経済成長率・政策効果を予測するものではありません。実際の政策判断に使わないでください。

## 使い方（発表前に読む・スライドには出ません）

1. 「準備」セルから順に**すべてのセルを実行**する（グラフのコードは隠してあります）
2. **TODO** をチームの内容に書き換える。**実験していない結果・考察を書かない**
3. HTML スライド：`jupyter nbconvert notebooks/06_research_presentation.ipynb --to slides --no-input --execute`

## 準備：実験環境（最初に必ず実行）

G1 リポジトリの `.py`（計算の本体）を import できるように、作業ディレクトリ（G1 のルート）を自動で探して設定します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter / VSCode | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのリポジトリの URL** を入れる |
| Colab（方法B：ZIP をアップロード） | G1 フォルダを ZIP にして `/content` にアップロードしておく（またはセルの実行中にアップロード） |

Colab の CPU ランタイムで動きます（GPU・TPU は不要）。`textbook/`（教科書のコード）は公開リポジトリに含めない約束です。`textbook/` がないと import が失敗します。

In [ ]:
#@title 準備（G1 の .py を読み込む）
import os, sys, glob, zipfile, subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                                     # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "g1-economic-growth-optimization"   # リポジトリの中の G1 フォルダ（ルートが G1 なら ""）

IN_COLAB = "google.colab" in sys.modules


def is_g1_root(p):
    return (Path(p) / "economic_model.py").exists() and (Path(p) / "optimization.py").exists()


def find_g1_root():
    for p in [Path.cwd(), Path.cwd().parent]:          # ローカル：notebooks/ の中、または G1 のルート
        if is_g1_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    if REPO_URL:                                        # Colab 方法A
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_g1_root(cand):
            return cand
    zips = glob.glob("/content/*.zip")                  # Colab 方法B
    if not zips:
        from google.colab import files
        print("G1 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/g1_upload")
    for hit in glob.glob("/content/g1_upload/**/economic_model.py", recursive=True):
        if is_g1_root(Path(hit).parent):
            return Path(hit).parent
    return None


G1_ROOT = find_g1_root()
if G1_ROOT is None:
    raise RuntimeError("G1 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(G1_ROOT)
if str(G1_ROOT) not in sys.path:
    sys.path.insert(0, str(G1_ROOT))
if not (G1_ROOT / "textbook" / "fixed_queue.py").exists():
    print("⚠️ textbook/ がありません。economic_model.py・optimization.py の import が失敗します。")

# ---- 既存の .py（計算の本体）と、グラフ ----
import matplotlib.pyplot as plt
import economic_model as em
import simulation as sim
import allocation_strategies as al
import optimization as op
import evaluation as ev
import growth_charts as gc

gc.setup_style()                        # 文字を大きくするなら gc.setup_style(1.3)
PARAMS = em.load_economy()              # 架空の国家のパラメータ
SCENARIOS = al.load_scenarios()         # 比較用の配分シナリオ
LABELS = {sc["id"]: sc["label"] for sc in SCENARIOS}
YEARS = PARAMS["simulation_years"]
STEP = 0.1                              # 最適化の配分の刻み（10%）
M = al.units_count(STEP)

SAVE_FIGURES = False                    # True にすると、グラフを notebooks/figures/ に PNG で保存する
FIG_DIR = "notebooks/figures"


def show(fig, name):
    """グラフを表示し、SAVE_FIGURES = True なら PNG も保存する"""
    if SAVE_FIGURES:
        print("saved:", gc.save(fig, f"{FIG_DIR}/{name}.png"))
    plt.show()


def table(rows, columns):
    """辞書のリストを表で表示する。columns = [(キー, 見出し, 書式)]。pandas があれば DataFrame で表示"""
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame([{h: ("-" if r.get(k) is None else f.format(r[k])) for k, h, f in columns} for r in rows]))
    except ImportError:
        print(ev.format_table(rows, columns))


def alloc_cols():
    return [(s, em.SECTOR_LABELS[s], "{:.0%}") for s in em.SECTORS]


print("G1_ROOT:", G1_ROOT)
print(PARAMS["country"], "｜", YEARS, "年、年間予算", PARAMS["annual_investment_budget"], "（架空の単位）")
print("シナリオ:", [sc["label"] for sc in SCENARIOS])

## 研究背景

- TODO：なぜ公共投資の配分をアルゴリズムで考えるのか

## Research Question

- TODO：チームが選んだ RQ（RQ1〜RQ6 から）

## 仮説

- TODO：**実験の前に**立てた仮説と、その理由

## 経済成長モデル

- 投資 → 状態変数（人的資本・技術・インフラ・健康）→ GDP・生活水準
- 投資効果 = 効率 × 投資額$^{0.5}$（収穫逓減）、教育・研究開発は**遅延**あり、状態変数は**減耗**する
- GDP = 初期 GDP × 人口比 × $H^{0.25} A^{0.30} I^{0.25} S^{0.15}$、生活水準 = 1 人あたり GDP・健康・人的資本の加重和
- **すべて架空の値**

In [ ]:
#@title 状態変数の時系列
show(gc.plot_state_panels(sim.simulate(al.equal_allocation(), PARAMS), "Equal allocation (model)"), "06_states")

## 投資配分モデル

- 年間予算 100（架空の単位）を 4 分野に配分：各 0〜1、合計 1
- 比較シナリオ：均等配分・教育重視・研究開発重視・インフラ重視・医療重視・アルゴリズムによる最適配分

In [ ]:
#@title 配分シナリオ
show(gc.plot_allocation_bars([(LABELS[sc["id"]].replace(" ", "\n", 1), sc["shares"]) for sc in SCENARIOS],
                             "Allocation scenarios"), "06_allocation")

## 最適化アルゴリズム

| | 全探索 | Greedy 法 | 局所探索 |
| --- | --- | --- | --- |
| 評価回数 | $C(M+3, 3)$ | $4M$ | 反復 × 近傍 |
| 最適性 | 離散空間で保証 | 保証なし | 保証なし（局所最適） |

## 実験条件

- TODO：目的関数・評価期間・刻み・変えたパラメータをチームの実験に合わせて書く

In [ ]:
#@title 実験条件
OBJECTIVE = "final_gdp"           # TODO：チームの実験条件にする
runs = sim.run_scenarios(SCENARIOS, PARAMS)
res = op.compare_algorithms(PARAMS, OBJECTIVE, STEP)
print(f"目的関数 {op.OBJECTIVE_LABELS[OBJECTIVE]}、評価期間 {YEARS} 年、刻み {STEP:.0%}（候補 {op.count_allocations(M)} 通り）")

## 結果：GDP 成長

In [ ]:
#@title GDP 成長
show(gc.plot_series(runs, "gdp", "GDP by scenario (model)", "GDP (fictional units)", labels=LABELS), "06_gdp")

## 結果：生活水準

In [ ]:
#@title 生活水準
show(gc.plot_series(runs, "living_standard", "Living standard by scenario (model)", labels=LABELS), "06_ls")

## 結果：最適化アルゴリズムの比較

In [ ]:
#@title アルゴリズムの比較
table([{"name": r["algorithm"], "objective": r["objective"], "gap": r["gap"]["absolute"], "runtime_ms": r["runtime_s"] * 1e3,
        "evaluations": r["evaluations"], **r["shares"]} for r in res.values()],
      [("name", "algorithm", "{}"), ("objective", "objective", "{:.3f}"), ("gap", "gap", "{:.4f}"),
       ("runtime_ms", "runtime (ms)", "{:.1f}"), ("evaluations", "evaluations", "{}")] + alloc_cols())
show(gc.plot_algorithm_metric(res, "evaluations", "Evaluated allocations by algorithm", "evaluations", fmt="{:d}"),
     "06_evaluations")

## 結果：感度分析

- TODO：チームの感度分析のグラフに置き換える

In [ ]:
#@title 感度分析
# TODO：チームの感度分析に置き換える（例：評価期間と最適配分）
items = []
for years in sim.HORIZONS:
    items.append((f"{years}y", op.exhaustive(op.Evaluator(PARAMS, OBJECTIVE, years), M)["shares"]))
show(gc.plot_allocation_bars(items, f"Optimal allocation by horizon ({op.OBJECTIVE_LABELS[OBJECTIVE]})"), "06_sensitivity")

## 考察

- TODO：仮説は支持されたか。**どの目的関数・制約条件・仮定の下で**そう言えるか
- TODO：アルゴリズムの長所・短所（最適性・計算量・初期条件への依存）

## モデルの限界

- 教育用の**架空モデル**。現実の国家の GDP・成長率・政策効果を予測しない
- 投資効果の係数・遅延・減耗率は架空。投資配分と経済成長の関係は**モデルの仮定**に依存する
- GDP の最大化は社会的厚生の最大化を意味しない。生活水準は人工的な指標
- 教育・医療の社会的価値を GDP だけで評価しない。実際の政策判断に使わない
- 最適解は目的関数・制約条件・モデルの仮定によって変わる
- TODO：チームが気づいた限界を追加する

## 結論

- TODO：RQ への答えを、**仮定とともに**書く

## 今後の課題

- TODO：Advanced A〜G のどれに取り組むか（複数国家、世界全体の配分、技術波及、確率的ショック、動的配分、多目的最適化、公開統計との比較）

## 参考文献

- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第2章 最大値、第3章 線形探索、第4章 キュー、第6章 単純挿入ソート）
- G1 リポジトリの `README.md`（モデルの仮定・計算量・モデルの限界）
- データ：`data/baseline_economy.json`、`data/investment_scenarios.json`（この教材のための**架空のデータ**）